In [1]:
import json
from pathlib import Path
from typing import Dict, Any, List

import pandas as pd

In [2]:
RESULTS_ROOT = Path("results")
OUTPUT_CSV = Path("results_aggregated.csv")

FAILURE_KEYS = ["WA", "TLE", "COMPLEXITY", "RE", "exception"]

In [3]:
def extract_date_from_path(path: Path) -> str:
    """
    Extract date string from path like:
    results/20251218_063138/agent_ragk0_summary.json
    -> 20251218_063138
    """
    for part in path.parts:
        if "_" in part and part.replace("_", "").isdigit():
            return part
    return "unknown"


def load_summary(path: Path) -> Dict[str, Any]:
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)


def normalize_record(data: Dict[str, Any], date: str) -> Dict[str, Any]:
    failure_counts = data.get("failure_type_counts", {}) or {}

    row = {
        "date": date,
        "mode": data.get("mode", ""),
        "n": data.get("n", 0),
        "success": data.get("success", 0),
        "success_rate": data.get("success_rate", 0.0),
        "avg_success_steps": data.get("avg_success_steps", 0.0),
        "n_iter": data.get("pass_at", 0),
        "rag_k": data.get("rag_k", 0),
        "total_runtime_sec": data.get("total_runtime_sec", 0.0),
    }

    # Flatten failure counts
    for key in FAILURE_KEYS:
        row[key] = failure_counts.get(key, 0)

    return row

In [4]:
def aggregate_results(results_root: Path) -> pd.DataFrame:
    rows: List[Dict[str, Any]] = []

    for path in results_root.rglob("*_summary.json"):
        try:
            data = load_summary(path)
        except Exception as exc:
            print(f"[WARN] Failed to load {path}: {exc}")
            continue

        date = extract_date_from_path(path)
        row = normalize_record(data, date)
        rows.append(row)

    if not rows:
        raise RuntimeError("No *_summary.json files found.")

    df = pd.DataFrame(rows)

    # Ensure column order
    columns = [
        "date",
        "mode",
        "n",
        "success",
        "success_rate",
        "WA",
        "TLE",
        "COMPLEXITY",
        "RE",
        "exception",
        "avg_success_steps",
        "n_iter",
        "rag_k",
        "total_runtime_sec",
    ]
    df = df[columns]

    return df

In [5]:
df = aggregate_results(RESULTS_ROOT)
df.to_csv(OUTPUT_CSV, index=False)
print(f"[OK] Aggregated {len(df)} rows → {OUTPUT_CSV}")


[OK] Aggregated 38 rows → results_aggregated.csv
